In [1]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import os
os.environ["LOKY_MAX_CPU_COUNT"] = "8"   # nº de núcleos físicos da sua máquina

SEED = 42

digits = load_digits()
X = digits.data.astype(np.float64)   # (1797, 64)
y = digits.target

# 1. Padronizar (obrigatório: PCA é sensível à escala)
X_std = StandardScaler().fit_transform(X)

# 2. PCA completo apenas para ler a curva de variância
pca_full = PCA(random_state=SEED).fit(X_std)
var_acum = np.cumsum(pca_full.explained_variance_ratio_)
n90 = int(np.argmax(var_acum >= 0.90) + 1)
print(f"Componentes para 90% da variância: {n90} de {X.shape[1]}")

# 3. UM único PCA, sobre X_std — e este é o que segue para o t-SNE
pca = PCA(n_components=n90, random_state=SEED)
X_pca = pca.fit_transform(X_std)
print(f"Variância retida: {pca.explained_variance_ratio_.sum():.2%}")
print(f"Shape: {X.shape} -> {X_pca.shape}")

# 4. t-SNE sobre as componentes do PCA
tsne = TSNE(
    n_components=2,
    perplexity=30,
    init="pca",
    learning_rate="auto",
    random_state=SEED,
)
X_tsne = tsne.fit_transform(X_pca)
print(f"KL final: {tsne.kl_divergence_:.4f}")

Componentes para 90% da variância: 31 de 64
Variância retida: 90.04%
Shape: (1797, 64) -> (1797, 31)
KL final: 0.8293
